#### Comparação de Modelos

Este notebook tem por objetivo comparar o desempenho de diferentes modelos de machine learning, especificamente `LogisticRegression`, 
`RandomForestClassifier` e `MLPClassifier`.

##### 1. Configuração do ambiente

In [20]:
import json
from datetime import datetime

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 7
TEST_SIZE = 0.2


# Método utilitário para registro dos experimentos
def log_training_run(model_name, metrics: dict, log_filepath):
    run_log = {
        "timestamp": datetime.now().isoformat(),  # noqa: DTZ005
        "model": model_name,
        "metrics": metrics,
    }

    with open(log_filepath, "a") as f:
        f.write(json.dumps(run_log) + "\n")

##### 2. Carregamento dos dados pré-processados

In [21]:
df = pd.read_csv("../data/telco_customer_churn_preprocessed.csv")

##### 3. Engenharia de Atributos

In [22]:
df["avg_charge"] = (df.total_charges / df.tenure_months).fillna(0)
df["diff_from_avg_charge"] = df.avg_charge - df.monthly_charges
df["total_charges"] = np.log1p(df["total_charges"])

##### 4. Treinamento e avaliação dos modelos

In [23]:
num_features = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "cltv",
    "avg_charge",
    "diff_from_avg_charge",
]
cat_features = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "phone_service",
    "multiple_lines",
    "internet_service",
    "online_security",
    "online_backup",
    "device_protection",
    "tech_support",
    "streaming_tv",
    "streaming_movies",
    "contract",
    "paperless_billing",
    "payment_method",
]

# Preparação dos dados
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Pré-processamento
num_transformer = make_pipeline(StandardScaler())
cat_transformer = make_pipeline(
    OneHotEncoder(handle_unknown="infrequent_if_exist", drop="first")
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
    ]
)

# Definição dos modelos a serem comparados no CV
models = {
    "logistic_regression": LogisticRegression(
        random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "random_forest": RandomForestClassifier(
        random_state=RANDOM_STATE, class_weight="balanced"
    ),
    "mlp": MLPClassifier(random_state=RANDOM_STATE, max_iter=5000),
}

# Métricas a serem registradas
scoring_metrics = ["average_precision", "precision", "recall", "accuracy", "f1"]

for model_name, model in models.items():
    # Integração do Pipeline, definição do modelo
    pipeline = make_pipeline(preprocessor, model)

    # Validação cruzada estratificada
    skfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    cv_results = cross_validate(
        pipeline, X_train, y_train, cv=skfold, scoring=scoring_metrics
    )

    metrics_dict = {}
    print(f"=== {model_name} ===")
    for metric_name in scoring_metrics:
        scores = cv_results[f"test_{metric_name}"]
        mean_val = scores.mean().round(4)
        std_val = scores.std().round(4)

        metrics_dict[metric_name] = {"mean": mean_val, "std": std_val}

        print(f"{metric_name}: {mean_val:.4f} (+/- {std_val:.4f})")
    print()

    # Registra os resultados dos experimentos em
    log_training_run(
        model_name, metrics_dict, log_filepath="../models/log_training_runs.jsonl"
    )


=== logistic_regression ===
average_precision: 0.6786 (+/- 0.0105)
precision: 0.5303 (+/- 0.0156)
recall: 0.8187 (+/- 0.0091)
accuracy: 0.7591 (+/- 0.0125)
f1: 0.6435 (+/- 0.0131)

=== random_forest ===
average_precision: 0.6415 (+/- 0.0113)
precision: 0.5897 (+/- 0.0160)
recall: 0.6716 (+/- 0.0086)
accuracy: 0.7886 (+/- 0.0082)
f1: 0.6278 (+/- 0.0091)

=== mlp ===
average_precision: 0.5572 (+/- 0.0161)
precision: 0.5518 (+/- 0.0241)
recall: 0.5144 (+/- 0.0166)
accuracy: 0.7599 (+/- 0.0109)
f1: 0.5321 (+/- 0.0154)



Considerando o problema de negócio trabalhado e a distribuição dos dados da classe que objetivamos prever, torna-se necessário encontrarmos com assertividade o maior número de clientes com probabilidade de churn. Nesse contexto, o modelo de regressão logística apresentou as melhores métricas de `recall` (0.8187) e `average_precision` (0.6786) dentre todos os modelos avaliados, e portanto foi escolhido como `champion`.

Abaixo reportamos suas métricas finais sendo treinado com todo o conjunto de treinamento e avaliado no conjunto de teste.

In [24]:
model_name = "logistic_regression_champion"
model = LogisticRegression(random_state=RANDOM_STATE, class_weight="balanced")

pipeline = make_pipeline(preprocessor, model)

pipeline.fit(X_train, y_train)

y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

average_precision = average_precision_score(y_test, y_proba)
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="binary"
)

metrics_dict = {
    "average_precision": round(average_precision, 4),
    "precision": round(precision, 4),
    "recall": round(recall, 4),
    "accuracy": round(accuracy, 4),
    "f1": round(f1, 4),
}
print(f"=== {model_name} ===")
for metric_name, metric_value in metrics_dict.items():
    print(f"{metric_name}: {metric_value}")

log_training_run(
    model_name, metrics_dict, log_filepath="../models/log_training_runs.jsonl"
)

=== logistic_regression_champion ===
average_precision: 0.7189
precision: 0.5447
recall: 0.8316
accuracy: 0.7708
f1: 0.6582


##### 5. Persistência do modelo campeão

In [25]:
joblib.dump(pipeline, "../models/champion_logistic_regression_pipeline.joblib")

['../models/champion_logistic_regression_pipeline.joblib']